# Analytical Placement — interactive training

Direct gradient descent on macro positions with a smooth differentiable proxy.

- **Live plot every 5 steps**: macro layout + loss curves.
- **Checkpoint every epoch**: saves positions + state to `checkpoints/`.
- Loss = smooth HPWL + ReLU overlap + canvas hinge (see `losses.py`).

## 1. Setup

Detects the best available device (CUDA → MPS → CPU). On an Apple Silicon Mac this picks `mps`. On an Intel Mac Pro it falls back to CPU and uses all cores.

In [ ]:
import os
import sys
from pathlib import Path

# Enable CPU fallback for any op MPS doesn't support yet.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Resolve the repo root so we can import `macro_place` and `submissions.analytical`.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "macro_place").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

ANALYTICAL_DIR = REPO_ROOT / "submissions" / "analytical"
CHECKPOINT_DIR = ANALYTICAL_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

import torch

def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = select_device()

# CPU-side perf: use every available core for the (small) Python-side work.
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)

print(f"Repo root: {REPO_ROOT}")
print(f"Device:    {DEVICE}")
print(f"Threads:   {torch.get_num_threads()}")

## 2. Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import clear_output

from macro_place.loader import load_benchmark_from_dir
from macro_place.objective import compute_proxy_cost
from submissions.analytical import losses as L

## 3. Load a benchmark

Edit `BENCHMARK_NAME` to switch designs.

In [ ]:
BENCHMARK_NAME = "ibm01"
BENCHMARK_DIR = REPO_ROOT / "external" / "MacroPlacement" / "Testcases" / "ICCAD04" / BENCHMARK_NAME

benchmark, plc = load_benchmark_from_dir(str(BENCHMARK_DIR))
print(benchmark)

## 4. Visualization helpers

`plot_placement` draws the macros on the canvas; `plot_loss` draws the loss curves. Both write into provided axes so we can compose a single figure in the live plot.

In [ ]:
def plot_placement(ax, positions, benchmark, title=""):
    """Draw hard and soft macros as rectangles on the canvas."""
    ax.clear()
    ax.set_xlim(0, benchmark.canvas_width)
    ax.set_ylim(0, benchmark.canvas_height)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")

    pos = positions.detach().cpu().numpy()
    sizes = benchmark.macro_sizes.cpu().numpy()
    fixed = benchmark.macro_fixed.cpu().numpy()
    num_hard = benchmark.num_hard_macros

    for i in range(num_hard):
        x = pos[i, 0] - sizes[i, 0] / 2.0
        y = pos[i, 1] - sizes[i, 1] / 2.0
        color = "lightgray" if fixed[i] else "steelblue"
        ax.add_patch(mpatches.Rectangle(
            (x, y), sizes[i, 0], sizes[i, 1],
            facecolor=color, edgecolor="black",
            alpha=0.55, linewidth=0.4,
        ))


def plot_loss(ax, history, log_y=True):
    """Plot smooth loss components over time."""
    ax.clear()
    if not history["step"]:
        return
    ax.plot(history["step"], history["wl"],      label="smooth WL")
    ax.plot(history["step"], history["overlap"], label="overlap")
    ax.plot(history["step"], history["total"],   label="total", linewidth=2)
    if log_y:
        ax.set_yscale("log")
    ax.set_xlabel("step")
    ax.set_ylabel("loss (log)" if log_y else "loss")
    ax.set_title("Smooth proxy loss")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

## 5. Training loop

Configuration knobs:

- `n_epochs` × `steps_per_epoch` = total optimizer steps.
- `plot_every` = 5: refresh the live plot every 5 steps.
- `checkpoint_every_epoch` = True: save positions to disk at every epoch boundary.
- `gamma` is annealed (sharper HPWL over time); `w_overlap` ramps up (zero-overlap by end).

In [ ]:
def train(
    benchmark,
    plc,
    *,
    n_epochs=20,
    steps_per_epoch=100,
    lr=5e-3,
    gamma_start=0.01,
    gamma_end=0.001,
    w_overlap_start=5.0,
    w_overlap_end=500.0,
    w_canvas=200.0,
    plot_every=5,
    checkpoint_dir=CHECKPOINT_DIR,
    device=DEVICE,
    seed=42,
):
    """Optimize macro positions with live tracing and per-epoch checkpoints."""
    torch.manual_seed(seed)
    total_steps = n_epochs * steps_per_epoch

    # ── Tensors on device ──────────────────────────────────────────────────
    positions = (
        benchmark.macro_positions.clone()
        .to(device, dtype=torch.float32)
        .requires_grad_(True)
    )
    sizes_dev = benchmark.macro_sizes.to(device, dtype=torch.float32)
    fixed_dev = benchmark.macro_fixed.to(device)
    anchor    = benchmark.macro_positions.to(device, dtype=torch.float32)
    padded_nets, mask, net_weights = L.prepare_net_tensors(benchmark, device)

    # Net nodes can contain port / macro-pin indices beyond num_macros.
    # Clamp them to a valid row and mask them out of the loss so they
    # contribute zero weight — equivalent to dropping them from each net.
    n_pos = positions.shape[0]
    out_of_bounds = padded_nets >= n_pos
    padded_nets = padded_nets.clamp(max=n_pos - 1)
    mask = mask & ~out_of_bounds

    optimizer = torch.optim.Adam([positions], lr=lr)
    history = {"step": [], "wl": [], "overlap": [], "canvas": [], "total": []}
    epoch_log = []  # list of dicts, one per epoch boundary

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    start_time = time.perf_counter()

    def linear(s, e, frac):
        return s + (e - s) * max(0.0, min(1.0, frac))

    for step in range(1, total_steps + 1):
        frac = step / total_steps
        gamma = linear(gamma_start, gamma_end, frac)
        w_overlap = linear(w_overlap_start, w_overlap_end, frac)

        optimizer.zero_grad()
        loss, comp = L.total_loss(
            positions, benchmark, padded_nets, mask, net_weights, sizes_dev,
            gamma=gamma, w_overlap=w_overlap, w_canvas=w_canvas,
        )
        loss.backward()

        # Pin fixed macros: zero their gradient and snap them back.
        with torch.no_grad():
            positions.grad[fixed_dev] = 0.0
        optimizer.step()
        with torch.no_grad():
            positions.data[fixed_dev] = anchor[fixed_dev]

        # ── Record history ──────────────────────────────────────────────────
        history["step"].append(step)
        history["wl"].append(comp["wl"])
        history["overlap"].append(max(comp["overlap"], 1e-12))
        history["canvas"].append(max(comp["canvas"], 1e-12))
        history["total"].append(comp["total"])

        # ── Live plot every `plot_every` steps ─────────────────────────────
        if step % plot_every == 0 or step == total_steps:
            clear_output(wait=True)
            plot_placement(
                axes[0], positions, benchmark,
                title=f"{benchmark.name}  step {step}/{total_steps}  γ={gamma:.4f}  wλ={w_overlap:.1f}",
            )
            plot_loss(axes[1], history)
            fig.tight_layout()
            plt.show()
            elapsed = time.perf_counter() - start_time
            print(
                f"step {step:>5d}/{total_steps}  "
                f"total={comp['total']:.3f}  "
                f"wl={comp['wl']:.3f}  "
                f"overlap={comp['overlap']:.4f}  "
                f"canvas={comp['canvas']:.4f}  "
                f"elapsed={elapsed:.1f}s"
            )

        # ── Checkpoint at each epoch boundary ──────────────────────────────
        if step % steps_per_epoch == 0:
            epoch_idx = step // steps_per_epoch
            ckpt_path = checkpoint_dir / f"{benchmark.name}_epoch{epoch_idx:03d}.pt"
            torch.save({
                "benchmark": benchmark.name,
                "epoch": epoch_idx,
                "step": step,
                "positions": positions.detach().cpu(),
                "optimizer_state": optimizer.state_dict(),
                "loss_components": comp,
                "history": history,
                "config": {
                    "lr": lr, "gamma": gamma, "w_overlap": w_overlap,
                    "w_canvas": w_canvas, "seed": seed,
                },
            }, ckpt_path)
            epoch_log.append({"epoch": epoch_idx, "step": step, "path": str(ckpt_path), **comp})

    plt.close(fig)
    return positions.detach().cpu(), history, epoch_log

## 6. Run training

In [ ]:
positions, history, epoch_log = train(
    benchmark,
    plc,
    n_epochs=20,
    steps_per_epoch=100,
    plot_every=5,
)

print("\nCheckpoints written:")
for row in epoch_log[-5:]:
    print(f"  epoch {row['epoch']:>3d}  total={row['total']:.3f}  {row['path']}")

## 7. Evaluate with the real TILOS proxy

Optimization was against the *smooth* proxy. This cell snaps the result
with the production placer's legalization pass and scores it with the
non-differentiable TILOS evaluator — the actual competition metric.

In [ ]:
from submissions.analytical.placer import AnalyticalPlacer
from submissions.core.eval import evaluate, visualize

# Use the production placer's legalization to guarantee zero overlaps.
placer = AnalyticalPlacer(n_steps=0, legalize_iters=300, device=DEVICE)
positions_legal = positions.clone().to(DEVICE).requires_grad_(True)
sizes_dev = benchmark.macro_sizes.to(DEVICE, dtype=torch.float32)
fixed_dev = benchmark.macro_fixed.to(DEVICE)
positions_legal = placer._clamp_to_canvas(positions_legal, sizes_dev, benchmark)
positions_legal = placer._restore_fixed(positions_legal, benchmark, fixed_dev)
positions_legal = placer._legalize_push_apart(positions_legal, sizes_dev, benchmark, fixed_dev)
positions_legal = placer._restore_fixed(positions_legal, benchmark, fixed_dev).detach().cpu()

result = evaluate(positions_legal, benchmark, plc)
print(
    f"Proxy: {result['proxy_cost']:.4f}  "
    f"WL: {result['wirelength_cost']:.4f}  "
    f"Density: {result['density_cost']:.4f}  "
    f"Congestion: {result['congestion_cost']:.4f}  "
    f"Overlaps: {result['overlap_count']}"
)

# Save the final legal placement and a visualization.
final_path = CHECKPOINT_DIR / f"{benchmark.name}_final.pt"
torch.save({"positions": positions_legal, "tilos": result}, final_path)
visualize(positions_legal, benchmark, plc=plc, save_path=str(CHECKPOINT_DIR / f"{benchmark.name}_final.png"))
print(f"Saved final placement -> {final_path}")

## 8. Resume from a checkpoint (optional)

Drop a checkpoint path here to warm-start a new training run from where
you left off — useful for tuning hyperparameters without losing progress.

In [ ]:
# Example: resume
# ckpt = torch.load(CHECKPOINT_DIR / f"{BENCHMARK_NAME}_epoch020.pt", weights_only=False)
# warm_positions = ckpt["positions"]
# print(f"Resuming from step {ckpt['step']}, total loss {ckpt['loss_components']['total']:.3f}")